# LLM-as-a-Judge

## নোটবুক পরিচিতি

একটি "judge" স্কোরিং ফাংশনের একটি শূন্য থেকে তৈরি, সম্পূর্ণ synthetic simulation
যেখানে একটি লুকানো (hidden) true-quality সংকেতের উপরে THREEটি ইচ্ছাকৃত,
পরিমাপযোগ্য bias বসানো আছে:

1. POSITION bias    — prompt-এর প্রথম slot-এ দেখানো response-টিকে দেওয়া একটি flat bonus।
2. VERBOSITY bias   — প্রকৃত গুণমান নির্বিশেষে, response-এর দৈর্ঘ্যের সমানুপাতিক bonus।
3. SELF-PREFERENCE  — judge-এর নিজের "model family" ভাগ করা response-কে দেওয়া bonus,
   bias               প্রকৃত গুণমান নির্বিশেষে।

নিচের প্রতিটি demo হাজার হাজার simulated pairwise comparison চালায় এবং bias-এর
win rate-এর উপর REAL পরিমাপিত প্রভাব রিপোর্ট করে, তারপর README-এর standard
mitigation প্রয়োগ করে REAL পুনরুদ্ধারকৃত win rate রিপোর্ট করে। কোথাও কোনো
LLM API call করা হয় না — এখানে "judge" অর্থ একটি ছোট স্কোরিং ফাংশন যাতে bias
ইনজেক্ট করা, যা একটি বাস্তব biased LLM judge-এর স্থানে দাঁড়িয়ে।

## কীভাবে চালাবেন

মূল file-টি হলো `example.py` — `python example.py` দিয়ে চলে। notebook-এ একই
কোড cell-by-cell চালানো হয়; শেষ cell-টি `main()` কল করে।

In [ ]:
import random

random.seed(0)

## Judge-এর raw স্কোরিং ফাংশন

লুকানো true quality + ইনজেক্ট করা bias + noise — এই ফাংশনটি একটি single
response-এর score তৈরি করে, যেমনটি একটি biased judge করত।

In [ ]:
def judge_score(quality, length, shown_first, judge_family, response_family,
                 position_bonus=0.0, verbosity_coef=0.0, self_pref_bonus=0.0,
                 noise_std=1.0):
    """একটি response-এর score, যেমনটি একটি biased judge তৈরি করবে।
    `quality` হলো response-এর HIDDEN ground-truth ভালোমন্দ — এমন জিনিস
    যা কেবল এই simulation জানে, judge নিজে কখনো নয়।"""
    score = quality
    if shown_first:
        score += position_bonus
    score += verbosity_coef * length
    if response_family == judge_family:
        score += self_pref_bonus
    score += random.gauss(0.0, noise_std)
    return score

## Demo 1: POSITION BIAS

একটি harness যা সবসময় "New" মডেলের response-কে slot 1-এ এবং "Baseline"-এর
response-কে slot 2-এ দেখায় — eval script লেখার একটি খুব স্বাভাবিক (এবং খুব
সাধারণ) উপায়। দুটি মডেলই অভিন্ন (IDENTICAL) true quality distribution থেকে
আঁকা, তাই একটি ন্যায্য judge-এর New-এর win ~50% পাওয়ার কথা।

In [ ]:
def position_bias_demo():
    print("=" * 78)
    print("1. POSITION BIAS: same quality distribution, fixed slot order")
    print("=" * 78)

    n_trials = 5000
    position_bonus = 2.0
    noise_std = 1.5

    print(f"New and Baseline responses both drawn from the SAME true-quality")
    print(f"distribution (Normal(mean=6, std=1)) -- a fair judge should find New")
    print(f"wins essentially 50% of the time. position_bonus={position_bonus}, noise_std={noise_std}\n")

    naive_new_wins = 0
    mitigated_new_wins = 0
    mitigated_ties = 0

    for _ in range(n_trials):
        quality_new = random.gauss(6.0, 1.0)
        quality_base = random.gauss(6.0, 1.0)
        length_new = random.gauss(100, 15)
        length_base = random.gauss(100, 15)

        # --- naive harness: New সবসময় প্রথমে দেখানো হয় ---
        score_new_naive = judge_score(quality_new, length_new, shown_first=True,
                                       judge_family="J", response_family="J",
                                       position_bonus=position_bonus, noise_std=noise_std)
        score_base_naive = judge_score(quality_base, length_base, shown_first=False,
                                        judge_family="J", response_family="J",
                                        position_bonus=position_bonus, noise_std=noise_std)
        if score_new_naive > score_base_naive:
            naive_new_wins += 1

        # --- mitigated: দুটি ordering-ই মূল্যায়ন, score পার্থক্যের average ---
        # ordering 1: New প্রথমে, Base দ্বিতীয়
        s_new_1 = judge_score(quality_new, length_new, shown_first=True,
                               judge_family="J", response_family="J",
                               position_bonus=position_bonus, noise_std=noise_std)
        s_base_1 = judge_score(quality_base, length_base, shown_first=False,
                                judge_family="J", response_family="J",
                                position_bonus=position_bonus, noise_std=noise_std)
        diff_1 = s_new_1 - s_base_1

        # ordering 2: Base প্রথমে, New দ্বিতীয়
        s_base_2 = judge_score(quality_base, length_base, shown_first=True,
                                judge_family="J", response_family="J",
                                position_bonus=position_bonus, noise_std=noise_std)
        s_new_2 = judge_score(quality_new, length_new, shown_first=False,
                               judge_family="J", response_family="J",
                               position_bonus=position_bonus, noise_std=noise_std)
        diff_2 = s_new_2 - s_base_2

        avg_diff = (diff_1 + diff_2) / 2.0
        if avg_diff > 0:
            mitigated_new_wins += 1
        elif avg_diff == 0:
            mitigated_ties += 1

    naive_rate = naive_new_wins / n_trials
    mitigated_rate = mitigated_new_wins / n_trials

    print(f"{'protocol':45}{'New win rate':>18}")
    print("-" * 63)
    print(f"{'naive (New always shown first)':45}{naive_rate:>18.1%}")
    print(f"{'mitigated (swap orderings, average diff)':45}{mitigated_rate:>18.1%}")

    print(f"\n-> New and Baseline have IDENTICAL true-quality distributions, so the")
    print(f"   correct win rate is 50%. The naive fixed-order harness measured")
    print(f"   {naive_rate:.1%} purely from the +{position_bonus} position bonus New always")
    print(f"   receives for sitting in the first slot -- New looks meaningfully better")
    print(f"   than Baseline even though it isn't. Swapping the order and averaging the")
    print(f"   two score differences brought the measured rate to {mitigated_rate:.1%}, within")
    print(f"   simulation noise of the true 50%, because the position bonus is added")
    print(f"   with opposite sign to New in the two orderings and cancels out exactly.")


position_bias_demo()

## Demo 2: VERBOSITY BIAS

"Concise" মডেলটি সত্যিই ভালো কিন্তু ছোট response লেখে; "Verbose" মডেলটি সত্যিই
খারাপ কিন্তু লম্বা লেখে। length-পুরস্কৃত করার term-যুক্ত একটি judge-কে খারাপ-কিন্তু-
লম্বা মডেলটি পছন্দ করতে বোকা বানানো যায়। নিচের cell-এ একটি closed-form OLS
ও আছে — judge-এর নিজস্ব length পছন্দ regress করে বের করতে।

In [ ]:
def linear_regression(xs, ys):
    """y ~ alpha + beta*x-এর জন্য closed-form OLS slope এবং intercept।"""
    n = len(xs)
    mean_x = sum(xs) / n
    mean_y = sum(ys) / n
    cov = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    var = sum((x - mean_x) ** 2 for x in xs)
    beta = cov / var
    alpha = mean_y - beta * mean_x
    return alpha, beta


def verbosity_bias_demo():
    print("\n" + "=" * 78)
    print("2. VERBOSITY BIAS: shorter-but-better vs. longer-but-worse")
    print("=" * 78)

    n_trials = 5000
    verbosity_coef = 0.018
    noise_std = 1.5

    print("Concise responses: true quality ~ Normal(7, 1), length ~ Normal(80, 10)")
    print("Verbose responses: true quality ~ Normal(5, 1), length ~ Normal(220, 20)")
    print(f"(Concise is genuinely 2 points better on average, but Verbose is far longer)")
    print(f"verbosity_coef={verbosity_coef}, noise_std={noise_std}, no position bias in this demo\n")

    oracle_wins = 0        # সিদ্ধান্ত শুধুমাত্র TRUE QUALITY (+ একই noise) দিয়ে -- সৎ baseline
    naive_wins = 0         # সিদ্ধান্ত verbosity-biased judge score দিয়ে
    corrected_wins = 0     # length effect regress করে বাদ দেওয়ার পর সিদ্ধান্ত

    records = []  # (score_concise, len_concise, score_verbose, len_verbose, oracle_win)

    for _ in range(n_trials):
        q_concise = random.gauss(7.0, 1.0)
        q_verbose = random.gauss(5.0, 1.0)
        len_concise = max(1.0, random.gauss(80, 10))
        len_verbose = max(1.0, random.gauss(220, 20))
        noise_c = random.gauss(0.0, noise_std)
        noise_v = random.gauss(0.0, noise_std)

        # Oracle: quality + noise-ই কেবল, verbosity term নেই -- "একটি ন্যায্য judge যা বলত"
        oracle_score_c = q_concise + noise_c
        oracle_score_v = q_verbose + noise_v
        oracle_win = oracle_score_c > oracle_score_v
        oracle_wins += int(oracle_win)

        # Naive judge: quality + verbosity bonus + একই noise draws (bias প্রভাবকে আলাদা করে)
        score_c = q_concise + verbosity_coef * len_concise + noise_c
        score_v = q_verbose + verbosity_coef * len_verbose + noise_v
        naive_win = score_c > score_v
        naive_wins += int(naive_win)

        records.append((score_c, len_concise, score_v, len_verbose))

    # judge-এর নিজস্ব প্রকাশিত length পছন্দকে ALL graded response-জুড়ে fit করি
    # (Concise ও Verbose — উভয়ের score ও length একত্রে pooled), তারপর
    # fit করা length effect-কে প্রতিটি score থেকে বাদ দিই।
    all_scores = [r[0] for r in records] + [r[2] for r in records]
    all_lengths = [r[1] for r in records] + [r[3] for r in records]
    alpha, beta = linear_regression(all_lengths, all_scores)

    for score_c, len_c, score_v, len_v in records:
        residual_c = score_c - (alpha + beta * len_c)
        residual_v = score_v - (alpha + beta * len_v)
        corrected_wins += int(residual_c > residual_v)

    oracle_rate = oracle_wins / n_trials
    naive_rate = naive_wins / n_trials
    corrected_rate = corrected_wins / n_trials

    print(f"Fitted judge length preference (pooled OLS): score ~ {alpha:.3f} + {beta:.5f} * length")
    print(f"(the TRUE injected coefficient was {verbosity_coef} -- the fit undershoots it, because")
    print(f"Concise responses are simultaneously SHORTER and HIGHER quality, so some of the")
    print(f"genuine quality signal looks, to a pooled regression, like 'a negative length")
    print(f"relationship' and partly masks the true verbosity effect being estimated)\n")

    print(f"{'protocol':45}{'Concise (truly better) win rate':>34}")
    print("-" * 79)
    print(f"{'oracle (true quality only, no bias)':45}{oracle_rate:>34.1%}")
    print(f"{'naive (verbosity-biased judge)':45}{naive_rate:>34.1%}")
    print(f"{'corrected (length regressed out)':45}{corrected_rate:>34.1%}")

    print(f"\n-> The oracle rate ({oracle_rate:.1%}) is what a judge with no verbosity bias")
    print(f"   would measure, using nothing but the true quality gap. The naive judge's")
    print(f"   length bonus for Verbose's much longer responses drags its measured rate")
    print(f"   for the ACTUALLY better model down to {naive_rate:.1%} -- a real, measurable")
    print(f"   distortion that makes the WORSE model win most comparisons. Regressing the")
    print(f"   judge's own scores on length and using the residual moves the rate back up")
    print(f"   to {corrected_rate:.1%} -- no longer favoring the worse model -- but it does NOT fully")
    print(f"   recover the {oracle_rate:.1%} oracle rate, because the pooled fit above underestimates")
    print(f"   the true bias coefficient. This is an honest, known limitation of naive length")
    print(f"   regression: when length and true quality are themselves correlated across the")
    print(f"   compared systems, a simple pooled correction only partially separates them --")
    print(f"   which is exactly why production length-controlled evaluators (e.g. AlpacaEval's")
    print(f"   length-controlled win rate) use more careful statistical controls than this.")


verbosity_bias_demo()

## Demo 3: SELF-PREFERENCE BIAS

"F1" family-র একটি single judge তার নিজের family-র সাথে মিলে যাওয়া response-কে
অনুকূল করে, এমনকি equal true quality-তেও। বিভিন্ন family-র judge-দের একটি panel,
গড়ে নিলে, এই idiosyncratic per-judge bias-কে দমন করে।

In [ ]:
def self_preference_bias_demo():
    print("\n" + "=" * 78)
    print("3. SELF-PREFERENCE BIAS: one judge vs. a diverse panel")
    print("=" * 78)

    n_trials = 5000
    self_pref_bonus = 1.8
    noise_std = 1.5
    judge_families = ["F1", "F2", "F3"]

    print("Response from family 'F1' vs. response from family 'F2', both drawn from")
    print("the SAME true-quality distribution -- a fair verdict should be ~50% either way.")
    print(f"Each judge gives a +{self_pref_bonus} bonus to a response sharing its OWN family.\n")

    single_judge_f1_wins = 0   # একাকী F1-family judge, F1 বনাম F2 গ্রেড করছে
    panel_f1_wins = 0          # 3-judge panel (F1, F2, F3), majority vote

    for _ in range(n_trials):
        quality_f1_response = random.gauss(6.0, 1.0)
        quality_f2_response = random.gauss(6.0, 1.0)
        length_f1 = random.gauss(100, 15)
        length_f2 = random.gauss(100, 15)

        # --- একটি single judge, family F1-এর সদস্য ---
        s_f1 = judge_score(quality_f1_response, length_f1, shown_first=True,
                            judge_family="F1", response_family="F1",
                            self_pref_bonus=self_pref_bonus, noise_std=noise_std)
        s_f2 = judge_score(quality_f2_response, length_f2, shown_first=False,
                            judge_family="F1", response_family="F2",
                            self_pref_bonus=self_pref_bonus, noise_std=noise_std)
        if s_f1 > s_f2:
            single_judge_f1_wins += 1

        # --- 3 জন judge-এর একটি panel (F1, F2, F3), প্রত্যেকের নিজস্ব
        #     self-preference, majority vote বিজয়ী নির্ধারণ করে ---
        votes_for_f1 = 0
        for judge_fam in judge_families:
            js_f1 = judge_score(quality_f1_response, length_f1, shown_first=True,
                                 judge_family=judge_fam, response_family="F1",
                                 self_pref_bonus=self_pref_bonus, noise_std=noise_std)
            js_f2 = judge_score(quality_f2_response, length_f2, shown_first=False,
                                 judge_family=judge_fam, response_family="F2",
                                 self_pref_bonus=self_pref_bonus, noise_std=noise_std)
            if js_f1 > js_f2:
                votes_for_f1 += 1
        if votes_for_f1 >= 2:   # majority of 3 (3-এর majority)
            panel_f1_wins += 1

    single_rate = single_judge_f1_wins / n_trials
    panel_rate = panel_f1_wins / n_trials

    print(f"{'protocol':45}{'F1-response win rate':>24}")
    print("-" * 69)
    print(f"{'single judge (family F1)':45}{single_rate:>24.1%}")
    print(f"{'panel of 3 judges (F1, F2, F3), majority vote':45}{panel_rate:>24.1%}")

    print(f"\n-> True quality is identical between the F1 and F2 responses, so 50% is the")
    print(f"   correct rate. The lone F1 judge, biased toward its own family, measured")
    print(f"   {single_rate:.1%}. Spreading the self-preference bias across a 3-judge panel")
    print(f"   from different families -- each pulling toward a DIFFERENT family -- brought")
    print(f"   the majority-vote rate to {panel_rate:.1%}, closer to the true 50%, because no")
    print(f"   single family's bias can dominate a majority vote across a diverse panel.")


self_preference_bias_demo()

## সবগুলো demo একসাথে: main()

`main()` তিনটি demo একই ক্রমে চালায় — মূল `example.py`-তে এটি
`if __name__ == "__main__":` guard-এর ভেতরে; notebook-এ শেষ cell হিসেবে `main()`
কল করা হয়।

In [ ]:
def main():
    position_bias_demo()
    verbosity_bias_demo()
    self_preference_bias_demo()


main()